# Módulo 0 — Registro de Usuarios y Captura de Dataset

Este cuaderno implementa el flujo descrito en la arquitectura para generar el dataset biométrico inicial:

- Solicitar datos del usuario (DNI, nombre, contraseña secundaria).
- Capturar imágenes durante 5 segundos, con opción de extender hasta cumplir el mínimo.
- Validar calidad (desenfoque, iluminación, posición, tamaño mínimo, presencia de gafas).
- Guardar únicamente las capturas válidas y registrar los eventos.

> **Nota:** Este módulo no genera embeddings; su función es exclusiva para adquisición y depuración de datos visuales.

## Secuencia Recomendada de Ejecución

1. Revisar y ajustar los parámetros de configuración.
2. Registrar la información del usuario (DNI, nombre, contraseña secundaria).
3. Validar la conectividad de la cámara y ejecutar las utilidades de diagnóstico.
4. Ejecutar la celda de captura para recolectar al menos 10 imágenes válidas.
5. Confirmar el resumen del proceso y validar que los archivos se hayan almacenado.

Cada bloque está modularizado para facilitar pruebas individuales y ajustes experimentales.

### Modo guiado recomendado

Para agilizar las pruebas puedes activar un perfil preconfigurado antes de capturar:

1. Ejecuta la celda **"Perfiles de captura"** (justo después de la configuración) para elegir entre `estricto`, `guiado` o `debug`.
2. Verifica el resumen impreso para saber cuántos segundos se capturarán, cuántos cuadros válidos se exigen y qué umbrales se usan.
3. Si necesitas ajustar un valor puntual, edítalo en el perfil activo y vuelve a aplicar el perfil.
4. El ejecutor intentará automáticamente con la cadena `guiado → debug` si no se alcanza el mínimo de frames; puedes personalizar los perfiles en la celda "Ejecutor con reintentos".

Así puedes arrancar rápidamente con parámetros tolerantes y luego volver al modo estricto cuando tus sujetos estén listos.

In [45]:
# --- Dependencias base ---
import os
import json
import time
import hashlib
import getpass

from pathlib import Path
from datetime import datetime, timezone
from typing import Any

import cv2
import numpy as np


In [25]:
# --- Configuración del entorno experimental ---
BASE_DIR = Path.cwd()
DATASET_DIR = BASE_DIR / "dataset_users"
LOGS_DIR = BASE_DIR / "logs"
REG_LOG_FILE = LOGS_DIR / "registration_log.json"
HAAR_CASCADE = cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
EYEGLASSES_CASCADE = cv2.data.haarcascades + "haarcascade_eye_tree_eyeglasses.xml"

CAPTURE_WINDOW_SECONDS = 5
MIN_VALID_FRAMES = 10
MAX_FRAME_BUFFER = 30  # límite de frames inspeccionados por sesión

QUALITY_CONFIG = {
    "blur_threshold": 110.0,
    "min_brightness": 70,
    "max_brightness": 220,
    "min_face_area": 9500,
    "max_faces": 1
}

LIVENESS_CONFIG = {
    "min_global_motion": 3.5,
    "min_eye_motion": 6.0,
    "initial_frames": 3
}

GLASSES_DETECTION_ENABLED = True

if os.name == "nt":
    CAMERA_BACKEND = cv2.CAP_DSHOW if hasattr(cv2, "CAP_DSHOW") else cv2.CAP_ANY
else:
    CAMERA_BACKEND = cv2.CAP_V4L2 if hasattr(cv2, "CAP_V4L2") else cv2.CAP_ANY

for directory in (DATASET_DIR, LOGS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

face_detector = cv2.CascadeClassifier(HAAR_CASCADE)
glasses_detector = cv2.CascadeClassifier(EYEGLASSES_CASCADE)
if face_detector.empty() or glasses_detector.empty():
    raise RuntimeError("No se pudo cargar el clasificador Haar requerido.")


In [26]:
# --- Perfiles de captura y diagnóstico rápido ---
import copy

CAPTURE_PROFILES = {
    "estricto": {
        "window": 5,
        "min_valid_frames": 10,
        "max_frame_buffer": 30,
        "quality": {
            "blur_threshold": 110.0,
            "min_brightness": 70,
            "max_brightness": 220,
            "min_face_area": 9500
        },
        "liveness": {
            "min_global_motion": 3.5,
            "min_eye_motion": 6.0,
            "initial_frames": 3
        },
        "glasses_block": True
    },
    "guiado": {
        "window": 8,
        "min_valid_frames": 6,
        "max_frame_buffer": 40,
        "quality": {
            "blur_threshold": 80.0,
            "min_brightness": 60,
            "max_brightness": 235,
            "min_face_area": 7500
        },
        "liveness": {
            "min_global_motion": 1.8,
            "min_eye_motion": 3.0,
            "initial_frames": 2
        },
        "glasses_block": False
    },
    "debug": {
        "window": 12,
        "min_valid_frames": 3,
        "max_frame_buffer": 60,
        "quality": {
            "blur_threshold": 40.0,
            "min_brightness": 40,
            "max_brightness": 255,
            "min_face_area": 5000
        },
        "liveness": {
            "min_global_motion": 0.5,
            "min_eye_motion": 0.8,
            "initial_frames": 1
        },
        "glasses_block": False
    }
}

ACTIVE_PROFILE = "guiado"


def apply_capture_profile(profile_name: str) -> None:
    if profile_name not in CAPTURE_PROFILES:
        raise ValueError(f"Perfil desconocido: {profile_name}")
    profile = CAPTURE_PROFILES[profile_name]
    globals()["CAPTURE_WINDOW_SECONDS"] = profile["window"]
    globals()["MIN_VALID_FRAMES"] = profile["min_valid_frames"]
    globals()["MAX_FRAME_BUFFER"] = profile["max_frame_buffer"]
    QUALITY_CONFIG.update(copy.deepcopy(profile["quality"]))
    LIVENESS_CONFIG.update(copy.deepcopy(profile["liveness"]))
    globals()["GLASSES_DETECTION_ENABLED"] = profile["glasses_block"]
    globals()["ACTIVE_PROFILE"] = profile_name


def describe_current_profile():
    print(f"Perfil activo: {ACTIVE_PROFILE}")
    print(
        f"- Ventana de captura: {CAPTURE_WINDOW_SECONDS}s | Frames válidos requeridos: {MIN_VALID_FRAMES}"
    )
    print(f"- Buffer máximo inspeccionado: {MAX_FRAME_BUFFER} frames")
    print("- Umbrales de calidad:")
    for key, value in QUALITY_CONFIG.items():
        print(f"    {key}: {value}")
    print("- Parámetros de liveness:")
    for key, value in LIVENESS_CONFIG.items():
        print(f"    {key}: {value}")
    print(f"- Bloqueo por gafas activado: {GLASSES_DETECTION_ENABLED}")


apply_capture_profile(ACTIVE_PROFILE)
describe_current_profile()
print("\nUsa apply_capture_profile('estricto'|'guiado'|'debug') para cambiar de perfil antes de capturar.")

Perfil activo: guiado
- Ventana de captura: 8s | Frames válidos requeridos: 6
- Buffer máximo inspeccionado: 40 frames
- Umbrales de calidad:
    blur_threshold: 80.0
    min_brightness: 60
    max_brightness: 235
    min_face_area: 7500
    max_faces: 1
- Parámetros de liveness:
    min_global_motion: 1.8
    min_eye_motion: 3.0
    initial_frames: 2
- Bloqueo por gafas activado: False

Usa apply_capture_profile('estricto'|'guiado'|'debug') para cambiar de perfil antes de capturar.


In [46]:
# --- Utilidades generales ---
def slugify(value: str) -> str:
    return "".join(ch for ch in value.lower().strip().replace(" ", "-") if ch.isalnum() or ch == "-")


def hash_password(password: str) -> str:
    return hashlib.sha256(password.encode("utf-8")).hexdigest()


def load_log() -> list:
    if REG_LOG_FILE.exists():
        with REG_LOG_FILE.open("r", encoding="utf-8") as f:
            return json.load(f)
    return []


def make_json_safe(value: Any):
    if isinstance(value, dict):
        return {k: make_json_safe(v) for k, v in value.items()}
    if isinstance(value, list):
        return [make_json_safe(v) for v in value]
    if isinstance(value, tuple):
        return [make_json_safe(v) for v in value]
    if isinstance(value, set):
        return [make_json_safe(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    return value


def append_log(entry: dict) -> None:
    data = load_log()
    data.append(entry)
    safe_data = make_json_safe(data)
    with REG_LOG_FILE.open("w", encoding="utf-8") as f:
        json.dump(safe_data, f, indent=2, ensure_ascii=False)


def build_user_paths(dni: str, name: str) -> dict:
    user_dir = DATASET_DIR / f"{dni}_{slugify(name)}"
    captures_dir = user_dir / "captures"
    meta_file = user_dir / "user_profile.json"
    user_dir.mkdir(parents=True, exist_ok=True)
    captures_dir.mkdir(parents=True, exist_ok=True)
    return {
        "root": user_dir,
        "captures": captures_dir,
        "meta": meta_file
    }


In [28]:
# --- Evaluación de calidad de imagen ---

def compute_laplacian_variance(frame: np.ndarray) -> float:
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()


def compute_brightness(frame: np.ndarray) -> float:
    hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
    return float(np.mean(hsv[:, :, 2]))


def detect_faces(frame: np.ndarray):
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(60, 60))
    return faces


def detect_eyeglasses(frame: np.ndarray) -> bool:
    if not GLASSES_DETECTION_ENABLED:
        return False
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    detections = glasses_detector.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=4, minSize=(40, 20))
    return len(detections) > 0


def assess_frame(frame: np.ndarray) -> dict:
    faces = detect_faces(frame)
    blur_score = compute_laplacian_variance(frame)
    brightness = compute_brightness(frame)
    has_goggles = detect_eyeglasses(frame)

    metrics = {
        "faces_detected": len(faces),
        "blur_score": blur_score,
        "brightness": brightness,
        "has_goggles": has_goggles,
        "face_areas": [w * h for (_, _, w, h) in faces]
    }

    validations = {
        "face_present": metrics["faces_detected"] == 1,
        "sharp_enough": blur_score >= QUALITY_CONFIG["blur_threshold"],
        "brightness_ok": QUALITY_CONFIG["min_brightness"] <= brightness <= QUALITY_CONFIG["max_brightness"],
        "face_size_ok": all(area >= QUALITY_CONFIG["min_face_area"] for area in metrics["face_areas"]) if metrics["face_areas"] else False,
        "glasses_block": not metrics["has_goggles"]
    }

    validations["frame_valid"] = all(validations.values())
    return {"metrics": metrics, "validations": validations}


In [29]:
# --- Registro de la identidad del usuario ---
user_dni = input("Ingrese DNI del usuario: ").strip()
if not user_dni:
    raise ValueError("El DNI no puede estar vacío.")

user_name = input("Ingrese nombre completo del usuario: ").strip()
if not user_name:
    raise ValueError("El nombre no puede estar vacío.")

password_plain = getpass.getpass("Ingrese contraseña secundaria: ")
if len(password_plain) < 4:
    raise ValueError("La contraseña secundaria debe tener al menos 4 caracteres.")

user_profile = {
    "dni": user_dni,
    "nombre": user_name,
    "password_hash": hash_password(password_plain),
    "created_at": datetime.now(timezone.utc).isoformat(),
    "captures": []
}

user_paths = build_user_paths(user_dni, user_name)
print(f"Directorio del usuario: {user_paths['captures']}")


Directorio del usuario: /home/andi/vXcode/Lofasys/dataset_users/76684955_anderson-quispe/captures


In [30]:
# --- Antispoofing (parpadeo y micro-movimientos) ---

def compute_motion_scores(current_gray: np.ndarray, previous_gray: np.ndarray) -> tuple[float, float]:
    if previous_gray is None:
        return 0.0, 0.0
    diff = cv2.absdiff(current_gray, previous_gray)
    global_motion = float(np.mean(diff))
    h = current_gray.shape[0]
    eye_region = current_gray[: h // 3, :]
    eye_prev = previous_gray[: h // 3, :]
    eye_diff = cv2.absdiff(eye_region, eye_prev)
    eye_motion = float(np.mean(eye_diff))
    return global_motion, eye_motion


def liveness_passed(global_motion: float, eye_motion: float, frame_index: int) -> bool:
    if frame_index < LIVENESS_CONFIG["initial_frames"]:
        return False
    return (
        global_motion >= LIVENESS_CONFIG["min_global_motion"]
        and eye_motion >= LIVENESS_CONFIG["min_eye_motion"]
    )


In [39]:
# --- Rutina principal de captura ---

def run_capture_session(user_profile: dict, user_paths: dict, show_preview: bool = True) -> dict:
    cap = cv2.VideoCapture(0, CAMERA_BACKEND)
    if not cap.isOpened():
        raise RuntimeError("No se pudo acceder a la cámara.")

    if show_preview:
        cv2.namedWindow("Captura guiada", cv2.WINDOW_NORMAL)
        cv2.resizeWindow("Captura guiada", 800, 600)

    start_time = time.time()
    frame_index = 0
    valid_frames = 0
    saved_samples = []
    rejection_notes = []
    previous_gray = None

    try:
        while frame_index < MAX_FRAME_BUFFER and (
            (time.time() - start_time) <= CAPTURE_WINDOW_SECONDS or valid_frames < MIN_VALID_FRAMES
        ):
            ret, frame = cap.read()
            if not ret:
                rejection_notes.append({"frame": frame_index, "reason": "Frame no capturado"})
                break

            if show_preview:
                display_frame = frame.copy()
                h, w, _ = display_frame.shape
                box_w, box_h = int(w * 0.6), int(h * 0.6)
                x1 = (w - box_w) // 2
                y1 = (h - box_h) // 2
                x2 = x1 + box_w
                y2 = y1 + box_h
                cv2.rectangle(display_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(
                    display_frame,
                    "Alinea tu rostro dentro del recuadro",
                    (max(10, x1), max(30, y1 - 10)),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.7,
                    (0, 255, 0),
                    2,
                    cv2.LINE_AA,
                )
                cv2.putText(
                    display_frame,
                    "Presiona 'q' para cancelar",
                    (10, h - 15),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (255, 255, 255),
                    1,
                    cv2.LINE_AA,
                )
                cv2.imshow("Captura guiada", display_frame)
                if cv2.waitKey(1) & 0xFF == ord("q"):
                    print("Captura cancelada por el usuario.")
                    break

            gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
            global_motion, eye_motion = compute_motion_scores(gray, previous_gray)
            previous_gray = gray

            assessment = assess_frame(frame)
            validations = assessment["validations"].copy()
            validations["liveness_ok"] = liveness_passed(global_motion, eye_motion, frame_index)
            validations["frame_valid"] = validations["frame_valid"] and validations["liveness_ok"]

            if validations["frame_valid"]:
                timestamp = datetime.now(timezone.utc).isoformat()
                filename = f"capture_{frame_index:03d}.jpg"
                filepath = user_paths["captures"] / filename
                cv2.imwrite(str(filepath), frame)
                saved_samples.append(
                    {
                        "file": filename,
                        "timestamp": timestamp,
                        "metrics": assessment["metrics"],
                        "motion": {"global": global_motion, "eyes": eye_motion}
                    }
                )
                valid_frames += 1
                print(f"[OK] Frame {frame_index} almacenado ({valid_frames}/{MIN_VALID_FRAMES})")
            else:
                failed_checks = [name for name, ok in validations.items() if name != "frame_valid" and not ok]
                rejection_notes.append({"frame": frame_index, "reason": failed_checks})
                print(f"[SKIP] Frame {frame_index} rechazado -> {failed_checks}")

            frame_index += 1

    finally:
        cap.release()
        if show_preview:
            cv2.destroyWindow("Captura guiada")
        cv2.destroyAllWindows()

    session_summary = {
        "valid_frames": valid_frames,
        "saved_samples": saved_samples,
        "rejections": rejection_notes,
        "start_time": start_time,
        "end_time": time.time()
    }

    return session_summary


In [40]:
# --- Utilidad para depurar rechazos ---
from collections import Counter

REJECTION_TIPS = {
    "face_present": "Mantén tu rostro centrado y evita que aparezcan otras personas en cuadro.",
    "sharp_enough": "Aumenta la luz o mantén la cabeza quieta un instante para reducir el desenfoque.",
    "brightness_ok": "Ajusta la iluminación o acércate a la fuente de luz para entrar en el rango permitido.",
    "face_size_ok": "Acércate un poco más a la cámara para que el rostro ocupe al menos el 60% del recuadro.",
    "glasses_block": "Retira las gafas si el perfil actual las bloquea o cambia al perfil guiado/debug.",
    "liveness_ok": "Parpadea y realiza micro-movimientos de cabeza para cumplir con liveness.",
    "frame_valid": "Verifica múltiples condiciones: rostro único, nitidez, brillo y liveness."
}


def summarize_rejections(rejections: list[dict]) -> Counter:
    counter = Counter()
    for entry in rejections:
        reason = entry.get("reason")
        if isinstance(reason, (list, tuple)):
            counter.update(reason)
        elif reason is not None:
            counter[str(reason)] += 1
    return counter


def print_rejection_report(rejections: list[dict]) -> None:
    counter = summarize_rejections(rejections)
    if not counter:
        print("Sin rechazos registrados: todas las capturas fueron válidas.")
        return
    print("\nMotivos de rechazo (conteo):")
    for reason, count in counter.most_common():
        print(f"- {reason}: {count}")
        tip = REJECTION_TIPS.get(reason)
        if tip:
            print(f"    ↳ Sugerencia: {tip}")

In [43]:
# --- Ejecutor con reintentos automáticos ---
AUTO_RETRY_ON_FAILURE = True
PROFILE_CHAIN = ["guiado", "debug"]
PREVIEW_ENABLED = True


def capture_with_failover(user_profile: dict, user_paths: dict,
                           profile_chain: list[str] | None = None,
                           auto_retry: bool = AUTO_RETRY_ON_FAILURE,
                           show_preview: bool = PREVIEW_ENABLED):
    sessions = []
    chain = profile_chain or PROFILE_CHAIN
    for idx, profile_name in enumerate(chain, start=1):
        if profile_name not in CAPTURE_PROFILES:
            raise ValueError(f"Perfil no definido: {profile_name}")
        apply_capture_profile(profile_name)
        required = MIN_VALID_FRAMES
        print(
            f"\n>>> Intento {idx}/{len(chain)} con perfil '{profile_name}' "
            f"(mínimo requerido: {required} frames válidos)"
        )
        session = run_capture_session(user_profile, user_paths, show_preview=show_preview)
        attempt_data = {
            "profile": profile_name,
            "required": required,
            "session": session
        }
        sessions.append(attempt_data)
        print_rejection_report(session["rejections"])
        if session["valid_frames"] >= required:
            return session, sessions
        if not auto_retry or idx == len(chain):
            break
        print("No se alcanzó el mínimo; se intentará con el siguiente perfil en 3 segundos...")
        time.sleep(3)
    return sessions[-1]["session"], sessions


In [48]:
# --- Ejecución controlada de la sesión de captura ---
profile_chain = PROFILE_CHAIN if AUTO_RETRY_ON_FAILURE else [ACTIVE_PROFILE]
session, attempts = capture_with_failover(
    user_profile,
    user_paths,
    profile_chain=profile_chain,
    auto_retry=AUTO_RETRY_ON_FAILURE,
    show_preview=PREVIEW_ENABLED
)

last_profile = attempts[-1]["profile"]
required_frames = attempts[-1]["required"]

if session["valid_frames"] < required_frames:
    raise RuntimeError(
        "Se capturaron {valid} frames válidos con el perfil '{profile}'; se requieren {req}.".format(
            valid=session["valid_frames"], req=required_frames, profile=last_profile
        )
    )

user_profile["captures"] = session["saved_samples"]
user_profile["capture_summary"] = {
    "total_valid": session["valid_frames"],
    "duration_seconds": session["end_time"] - session["start_time"],
    "dataset_path": str(user_paths["captures"]),
    "perfil_utilizado": last_profile
}

with user_paths["meta"].open("w", encoding="utf-8") as f:
    json.dump(make_json_safe(user_profile), f, indent=2, ensure_ascii=False)

append_log(
    {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "dni": user_profile["dni"],
        "modelo": "capture_module",
        "resultado": "exitoso",
        "valid_frames": session["valid_frames"],
        "dataset_path": str(user_paths["captures"]),
        "perfil": last_profile
    }
)

print("\nResumen de la sesión:")
print(json.dumps(user_profile["capture_summary"], indent=2, ensure_ascii=False))
print(
    "\nSi los rechazos persisten incluso tras el fallback automático, "
    "ajusta manualmente los parámetros o repite con iluminación distinta."
)



>>> Intento 1/2 con perfil 'guiado' (mínimo requerido: 6 frames válidos)
[SKIP] Frame 0 rechazado -> ['sharp_enough', 'liveness_ok']
[SKIP] Frame 1 rechazado -> ['sharp_enough', 'liveness_ok']
[SKIP] Frame 0 rechazado -> ['sharp_enough', 'liveness_ok']
[SKIP] Frame 1 rechazado -> ['sharp_enough', 'liveness_ok']
[SKIP] Frame 2 rechazado -> ['sharp_enough']
[SKIP] Frame 3 rechazado -> ['sharp_enough']
[SKIP] Frame 2 rechazado -> ['sharp_enough']
[SKIP] Frame 3 rechazado -> ['sharp_enough']
[SKIP] Frame 4 rechazado -> ['face_present', 'sharp_enough']
[SKIP] Frame 5 rechazado -> ['sharp_enough']
[SKIP] Frame 4 rechazado -> ['face_present', 'sharp_enough']
[SKIP] Frame 5 rechazado -> ['sharp_enough']
[SKIP] Frame 6 rechazado -> ['sharp_enough']
[SKIP] Frame 7 rechazado -> ['sharp_enough']
[SKIP] Frame 6 rechazado -> ['sharp_enough']
[SKIP] Frame 7 rechazado -> ['sharp_enough']
[SKIP] Frame 8 rechazado -> ['sharp_enough']
[SKIP] Frame 9 rechazado -> ['sharp_enough']
[SKIP] Frame 8 rechazado

### Próximos pasos

- Verificar manualmente las capturas en la carpeta del usuario y descartar manualmente cualquier anomalía.
- Ejecutar auditorías rápidas (visionado acelerado) para confirmar que no se almacenan frames con gafas o fallas de liveness.
- Continuar con los notebooks de modelo (`login_face_recognition`, `login_deepface`, `login_insightface`) para generar embeddings a partir del dataset recién creado.